# Phase 2: Text Data Preparation

This notebook prepares ticket descriptions and category labels for a later classification step. It does not create TF-IDF features or train a model.

We predict `Issue_Category` using `Ticket_Description`. We leave out `Ticket_Subject` because Phase 1 found its subject patterns strongly associated with the category, which could let a classifier use a shortcut rather than learn from the customer's description.

## 1. Load the raw dataset

**Why:** We start from the original data so each preparation step is clear and repeatable. The two CSV files in `data/raw/` were checked and are identical; this notebook uses `customer_support_tickets.csv` once.

In [ ]:
from pathlib import Path

import pandas as pd

data_path = Path("../data/raw/customer_support_tickets.csv")
tickets = pd.read_csv(data_path)

print(f"Raw dataset rows: {len(tickets)}")
tickets.head()

## 2. Choose input and target columns

**Why:** We keep only what is needed for the task: text available when a customer opens a ticket and the category label we want to predict. This also helps prevent unrelated or post-resolution information from reaching a future model.

- **Input text:** `Ticket_Description`
- **Target:** `Issue_Category`

We exclude `Resolution_Time_Hours` and `Satisfaction_Score` because they describe outcomes after the ticket is handled. We also exclude `Assigned_Agent`, which may be assigned after triage; `Ticket_ID`, `Customer_Name`, and `Customer_Email`, which are identifiers or personal details; and `Priority_Level`, `Ticket_Channel`, and `Submission_Date` to keep this first text-only dataset focused. `Ticket_Subject` is excluded because its wording appears strongly associated with the category in this dataset, so it may create a shortcut.

In [ ]:
text_column = "Ticket_Description"
target_column = "Issue_Category"

selected = tickets[[text_column, target_column]].copy()
selected.head()

## 3. Inspect missing values

**Why:** Missing text can be replaced with an empty string, but a missing target gives us no correct answer to learn from. We check both before cleaning.

In [ ]:
print("Missing values in selected columns:")
selected.isna().sum()

There are no missing values in these two columns in the current dataset. The next cells still handle missing data explicitly so the preparation remains understandable and works if the raw file changes.

## 4. Show examples before cleaning

**Why:** Looking at real examples makes it easier to see what the small cleaning steps will change.

In [ ]:
selected[[text_column, target_column]].head(5)

## 5. Handle missing values, duplicates, and clean text

**Why:** We fill missing descriptions with empty text, because the row may still have a useful category label. Rows without a target cannot be used for supervised learning, so we remove them. Exact duplicate input/target pairs can overrepresent the same example, so we keep only one copy.

Text cleaning is deliberately minimal: convert values to strings, trim leading and trailing whitespace, and collapse repeated whitespace. We preserve punctuation and the original wording because TF-IDF can use those words directly, and aggressive cleaning could remove useful clues.

In [ ]:
import re

rows_before_cleaning = len(selected)
cleaned = selected.copy()

# A row without a category has no label to learn from.
cleaned = cleaned.dropna(subset=[target_column])

# A missing description is treated as empty text.
cleaned[text_column] = cleaned[text_column].fillna("")

# Convert to strings, trim the ends, and normalize repeated whitespace.
cleaned["text"] = cleaned[text_column].astype(str).apply(
    lambda value: re.sub(r"\\s+", " ", value).strip()
)

# Keep one copy of each identical text and category pair.
cleaned = cleaned.drop_duplicates(subset=["text", target_column]).copy()

rows_after_cleaning = len(cleaned)
print(f"Rows before cleaning: {rows_before_cleaning}")
print(f"Rows after cleaning: {rows_after_cleaning}")
print(f"Rows removed: {rows_before_cleaning - rows_after_cleaning}")

## 6. Compare examples after cleaning

**Why:** Comparing the same examples confirms that we made only the intended whitespace changes and did not remove punctuation or words.

In [ ]:
comparison = selected[[text_column, target_column]].head(5).copy()
comparison["text_after_cleaning"] = cleaned["text"].head(5).values
comparison

## 7. Check the cleaned target distribution

**Why:** Cleaning or removing duplicates can change the number of examples in each category. We inspect the counts before splitting so the class balance is visible.

In [ ]:
target_distribution = cleaned[target_column].value_counts().rename("count").to_frame()
target_distribution["percent"] = (target_distribution["count"] / len(cleaned) * 100).round(1)
target_distribution

## 8. Prepare X and y

**Why:** In supervised classification, `X` is the information provided to the model and `y` is the answer it should learn to predict. Here, `X` is each cleaned ticket description and `y` is its issue category.

In [ ]:
X = cleaned["text"]
y = cleaned[target_column]

print(f"X rows: {len(X)}")
print(f"y rows: {len(y)}")
print("First X value:", X.iloc[0])
print("First y value:", y.iloc[0])

## 9. Make a stratified train/test split

**Why:** We separate training and test data so later we can fit a model on one portion and evaluate it on examples it did not train on. Stratification keeps approximately the same category proportions in both sets. This is useful here because `Fraud` has fewer examples than the other categories.

The split is 80% training and 20% testing, with `random_state=42` so the same split can be reproduced. No model is trained in this notebook.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

print(f"Training rows: {len(X_train)}")
print(f"Test rows: {len(X_test)}")

## 10. Save the cleaned dataset

**Why:** Saving this small table lets a later notebook load the prepared text and category without repeating the cleaning steps. It contains only the cleaned text feature and target label.

In [ ]:
output_path = Path("../data/processed/cleaned_tickets.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

cleaned[["text", target_column]].to_csv(output_path, index=False)
print(f"Saved {len(cleaned)} rows to {output_path}")

## Summary

- Text input: cleaned `Ticket_Description`, saved as `text`.
- Target: `Issue_Category`.
- `X` contains the text and `y` contains the category labels.
- The train/test split is prepared, but no model or TF-IDF features are created here.
- Minimal whitespace cleaning keeps the ticket wording intact for later TF-IDF processing.